# ICA

In [ ]:
subject_id = "101"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. derivatives_root -> a test folder
path_overrides = None


In [ ]:
import mne
from mne.preprocessing import ICA
import autoreject
from pathlib import Path
import pandas as pd
from datetime import datetime
import json
import os

from spectral.viz import plot_step, plot_bad_channels, plot_epochs
from spectral.specparam import specparam2pandas
from spectral.utils import ProjectPaths,print_timestamp
from spectral.ica import compute_ica_limited, plot_annotated_topographies, plot_component_properties,get_manual_ica_list
from mne_icalabel.gui import label_ica_components
from mne_icalabel import label_components


""" config = read_parameters()
my_paths = config_project(subject)
project_path = my_paths["project_path"]
figures_path = my_paths["figures_path"]
specparam_path = my_paths["specparam_path"]

Path(specparam_path).mkdir(parents=True, exist_ok=True)
 """


paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
# Create all directories
paths.create_directories()

#paths = ProjectPaths(subject)
# Create all directories
#paths.create_directories()

# This is analysis output, so it goes in the analysis folder
specparam_path = paths.specparam
project_path = paths.root
figures_path = paths.figures
preprocessed_path = paths.preprocessed
epochs_path = paths.epochs
# Print paths to verify
print_timestamp("Setting up project paths")
paths.show()

mne.viz.set_browser_backend("matplotlib")
# mne.viz.set_browser_backend("qt")
mne.set_config("MNE_BROWSER_THEME", "light")
# matplotlib.use("Agg")
#print_date_time()

In [ ]:
clean_raw = mne.io.read_raw_fif(
    f"{paths.preprocessed}/sub-{subject_id}_annotated_filtered_raw.fif", preload=True
)

total_duration = clean_raw.times[-1]
clean_raw = clean_raw.crop(tmin=3.0, tmax=total_duration - 3, include_tmax=True)

report = mne.Report(title=f"Report for subject {subject_id}", subject=subject_id)
report.add_raw(raw=clean_raw, title="Raw", psd=True, butterfly=True, scalings='auto')

# --- HANDLES CASE WITH NO BAD CHANNELS ---
if clean_raw.info["bads"]:
    print(f"Plotting {len(clean_raw.info['bads'])} bad channels for the report.")
    
    # We use a try-except here because plotting bads can occasionally 
    # throw a ValueError if channels were marked bad but then dropped.
    try:
        bad_channel_plot = clean_raw.plot(
            picks=clean_raw.info["bads"],
            duration=min(300.0, total_duration - 6), # Ensure duration isn't longer than file
            scalings=dict(eeg=1e-4),
            show_scrollbars=False,
            show=False # Prevent popup in headless environments
        )
        report.add_figure(bad_channel_plot, title="Rejected bad channels")
    except Exception as e:
        print(f"Could not plot bad channels: {e}")
else:
    print("No bad channels detected. Skipping bad channel plot.")
    # Optional: Add a text note to the report instead of a plot
    report.add_html(
        html="<p><b>Note:</b> No bad channels were detected or marked for this subject.</p>",
        title="Rejected bad channels"
    )

In [ ]:
epochs_good = mne.read_epochs(
    f"{paths.epochs}/sub-{subject_id}_good_epochs-epo.fif"
)

In [ ]:

# ICA parameters come from settings.toml ([cleaning]).
_cl = paths.config["cleaning"]
ica = compute_ica_limited(epochs_good,
                          variance_threshold=_cl["ica_variance_threshold"],
                          max_cap=_cl["ica_max_components"],
                          random_state=_cl["ica_random_state"])
print(ica.n_components_)
ica.save(
    f"{paths.analysis}/sub-{subject_id}_my_ica_model-ica.fif", overwrite=True)


In [ ]:
ica_plot_components = ica.plot_components()

In [ ]:
chunk_size = 10
chunks = [
    range(i, min(i + chunk_size, ica.n_components_))
    for i in range(0, ica.n_components_, chunk_size)
]

for chunk in chunks:
    ica_plot = ica.plot_sources(
        epochs_good,
        picks=chunk,
        show_scrollbars=False,
        start=0,
        stop=len(epochs_good) - 1,
    )
    # for chunk in chunks:
    ica_plot = ica.plot_sources(
        epochs_good,
        picks=chunk,
        show_scrollbars=False,
        start=0,
        stop=20,
    )

In [ ]:
explained_var_ratio = ica.get_explained_variance_ratio(epochs_good)
for channel_type, ratio in explained_var_ratio.items():
    print(
        f"Fraction of {
            channel_type} variance explained by all components: "
        f"{ratio}"
    )

# ICA label gui

# Removing components

In [ ]:
ecg_indices, ecg_scores = ica.find_bads_ecg(
    clean_raw, method="correlation", threshold="auto"
)

try:
    # ica_plot = ica.plot_sources(eeg_data)
    # Exclude blink artifact components (use Fp1 as EOG proxy)

    # ica.exclude = np.argwhere(abs(eog_scores) > 0.5).ravel().tolist()
    _ = ica.plot_sources(
        epochs_good,
        show_scrollbars=False,
        picks=ecg_indices,
        start=0,
        stop=300.0,
    )
    _ = ica.plot_properties(epochs_good, picks=ecg_indices)
except Exception as e:
    # If an exception occurs, print the error message
    print("An error occurred:", e)

# ICA components labelling 

In [ ]:
# 1. Classify
# Label on epochs_good -- the data the ICA was FITTED on. Passing clean_raw
# recomputes sources from time the decomposition never saw.
ic_labels = label_components(epochs_good, ica, method="iclabel")

# 2. Plot all topographies with their labels
plot_annotated_topographies(ica, clean_raw, ic_labels, report=report)

In [ ]:
all_ics = list(range(ica.n_components_))
# 2. Call your function with the full list
plot_component_properties(
    ica, 
    clean_raw, 
    picks=all_ics,        # This passes every component index
    ic_labels=ic_labels, 
    report=report,
    save_path=figures_path,
    subject=subject_id
)

# Select component to removal 

In [ ]:
# 3. Identify components to exclude (ECG hits + ICLabel artifacts)
# Per-class confidence thresholds come from settings.toml ([cleaning]); a flat
# 0.80 left blink components in for 6 of 20 subjects.
_thresholds = _cl.get("ica_label_thresholds", {
    "eye blink": 0.50, "muscle artifact": 0.80, "heart beat": 0.80,
    "line noise": 0.80, "channel noise": 0.80,
})
_ecg_reject_labels = _cl.get("ecg_reject_labels", ["brain"])

_labels, _probs = ic_labels["labels"], ic_labels["y_pred_proba"]

bad_indices = [
    i for i, label in enumerate(_labels)
    if label in _thresholds and _probs[i] > _thresholds[label]
]

# find_bads_ecg() returns whatever correlates with the ECG channel, brain
# components included. Keep only the hits ICLabel does not call brain.
ecg_kept = [i for i in ecg_indices if _labels[i] not in _ecg_reject_labels]
ecg_dropped = [i for i in ecg_indices if _labels[i] in _ecg_reject_labels]
if ecg_dropped:
    print("ECG hits ignored (ICLabel disagrees): "
          + ", ".join(f"IC{i} {_labels[i]} p={_probs[i]:.2f}" for i in ecg_dropped))

final_picks = sorted(set(bad_indices + ecg_kept))
print(f"excluding {len(final_picks)} of {ica.n_components_} components: "
      + ", ".join(f"IC{i} {_labels[i]} p={_probs[i]:.2f}" for i in final_picks))

# 4. Plot properties for ONLY the components we are worried about
plot_component_properties(
    ica, 
    clean_raw, 
    picks=final_picks, 
    ic_labels=ic_labels, 
    report=report,
    save_path=figures_path,
    subject=subject_id
)

# 5. Set the exclusion list
ica.exclude = final_picks


In [ ]:
# --- NEW: Plot Time Series (Sources) ---
if final_picks:
    print(f"Plotting time series for components: {final_picks}")
    # We plot a 20-second window by default to see the artifact patterns clearly
    fig_sources = ica.plot_sources(
        clean_raw, 
        picks=final_picks, 
        start=0, 
        stop=20, 
        show_scrollbars=False,
        show=True
    )
    
    if report is not None:
        report.add_figure(fig_sources, title="Excluded Components Time Series (Sources)")

## Save the decomposition and the decision file


In [ ]:
ica.save(f"{paths.analysis}/sub-{subject_id}_my_ica_model-ica.fif", overwrite=True)

# Per-subject decision file. Records the auto-detected components AND a
# fingerprint of this decomposition, so a manual selection made later can be
# checked (and topography-remapped) if the ICA is ever refitted.
# Manual fields are never written here -- they belong to the human.
from spectral.ica import write_ica_decision, ica_fingerprint

decision_path = paths.analysis / f"sub-{subject_id}_ica-decision.json"
decision = write_ica_decision(
    decision_path, ica,
    auto_detected=final_picks,
    labels={str(i): [ic_labels["labels"][i], float(ic_labels["y_pred_proba"][i])]
            for i in range(ica.n_components_)},
    extra={"subject_id": str(subject_id),
           "ica_params": paths.config["cleaning"],
           "ecg_indices": [int(i) for i in ecg_indices]},
)
print(f"decomposition fingerprint : {decision['ica_fingerprint']}")
print(f"auto-detected components  : {decision['auto_detected']}")
print(f"decision file             : {decision_path}")
print("\nTo override manually, review the component figures then run:")
print("    from spectral.ica import set_manual_exclusion")
print(f"    set_manual_exclusion(r'{decision_path}', ica, [0, 3, 7])")


In [ ]:
report.save(
    f"{paths.reports}/sub-{subject_id}_report_03_ica_fit.html", overwrite=True
)